<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Deep Learning for MNIST Classification</b></h1>
</div>

## Theoretical Foundations

This notebook formalizes the mathematical and algorithmic basis of the corresponding laboratory implementation. The section order mirrors the experimental workflow so that assumptions, estimation steps, diagnostics, and validation criteria remain directly traceable to the executable notebook.

### Technical Context

MNIST classification is supervised learning. For each image $\mathbf{x}$, the dataset provides a target digit $y\in\{0,\ldots,9\}$. The network learns a parameterized function

$$
f_{\theta}:\mathbb{R}^{784}\rightarrow\mathbb{R}^{10}
$$

that maps 784 normalized pixel values to 10 class scores called **logits**.

Training repeatedly performs

$$
\boxed{
\text{batch}
\rightarrow
\text{forward pass}
\rightarrow
\text{loss}
\rightarrow
\text{backpropagation}
\rightarrow
\text{parameter update}
}
$$

and evaluation performs

$$
\boxed{
\text{image}
\rightarrow
\text{logits}
\rightarrow
\text{Softmax}
\rightarrow
\text{class + confidence}
}
$$

### Core Neural Network Model

For each hidden width $H\in\{128,256,512\}$:

$$
784
\rightarrow
H
\rightarrow
10.
$$

More explicitly:

$$
\mathbf{x}
\rightarrow
\text{Linear}
\rightarrow
\text{BatchNorm}
\rightarrow
\text{ReLU}
\rightarrow
\text{Linear}
\rightarrow
\text{logits}.
$$

### Notation and Conventions

| Symbol | Meaning |
| --- | --- |
| $\mathbf{x}\in\mathbb{R}^{784}$ | one flattened normalized MNIST image |
| $y$ | ground-truth class label |
| $B$ | mini-batch size |
| $H$ | hidden-layer width |
| $W_1,b_1$ | first Linear layer parameters |
| $W_2,b_2$ | output Linear layer parameters |
| $\mathbf{z}_1$ | first-layer pre-activation |
| $\mathbf{h}$ | hidden representation after BatchNorm/ReLU |
| $\mathbf{z}\in\mathbb{R}^{10}$ | output logits |
| $p_k$ | Softmax probability of class $k$ |
| $\hat y$ | predicted class |
| $\mathcal{L}$ | Cross-Entropy loss |
| $\theta$ | all trainable model parameters |
| $\eta$ | learning rate |
| $TP,FP,FN$ | threshold-analysis counts |

### Analytical Scope

The analysis establishes the following points:

- why pixels are divided by 255;
- why MNIST is flattened for an MLP;
- what a Linear layer actually computes;
- why ReLU is necessary;
- what BatchNorm changes during training/evaluation;
- why Cross-Entropy expects logits rather than Softmax probabilities;
- how backpropagation and Adam modify weights;
- why hidden width affects capacity but not guarantees performance;
- how confidence differs from correctness;
- how thresholding changes precision and recall;
- why training loss alone is not sufficient evidence of generalization.

## 1. Validate MNIST Data and Output Paths

### Why data validation comes before deep learning

A neural network cannot compensate for a broken experiment setup. Before discussing gradients or architectures, we must know that the intended dataset is actually present.

MNIST is stored locally using four IDX files:

- training images;
- training labels;
- testing images;
- testing labels.

The image and label files must remain paired. If an image file and label file come from different datasets or are truncated, the network can train on meaningless supervision.

### Train set vs test set

The implementation uses

$$
N_{train}=60{,}000
$$

and

$$
N_{test}=10{,}000.
$$

The training set is used to update parameters. The test set must not participate in gradient updates.

This distinction is fundamental:

$$
\text{training data}
\Rightarrow
\text{learn parameters},
$$

$$
\text{test data}
\Rightarrow
\text{measure behavior after learning}.
$$

### Why repository-relative paths matter

A path such as

$$
\text{../data}
$$

makes the experiment reproducible from the notebook directory without depending on one user's absolute filesystem path.

The output directory is equally important because figures are part of the experimental evidence.

### Failure principle

A missing input should produce an explicit error instead of silently downloading another dataset or continuing with partial data.

That protects experiment identity.


## 2. Build the MNIST Dataset and Mini-Batch Pipeline

### How a digit becomes a model input

Each raw MNIST image contains

$$
28\times28=784
$$

grayscale pixel values.

The original values are bytes:

$$
x_j\in\{0,1,\ldots,255\}.
$$

The implementation converts them to floating point and normalizes:

$$
x_j^{norm}
=
\frac{x_j}{255}.
$$

Therefore

$$
x_j^{norm}\in[0,1].
$$

### Why normalize pixels?

Without normalization, input magnitudes can reach 255. With normalization, inputs are order one.

This usually improves numerical behavior during optimization because weight updates do not have to compensate for unnecessarily large input scale.

Normalization here is **not standardization**. We are not subtracting the dataset mean and dividing by standard deviation. We are simply rescaling bytes to $[0,1]$.

### Why flatten the image?

A fully connected MLP expects a vector. Therefore

$$
28\times28
\rightarrow
784.
$$

Conceptually:

$$
\mathbf{x}
=
[x_1,x_2,\ldots,x_{784}]^T.
$$

Flattening preserves all pixel values but removes explicit 2D neighborhood structure.

That is one reason an MLP is less image-aware than a CNN.

### Dataset abstraction

A PyTorch Dataset answers two questions:

1. **How many samples exist?**
2. **What is sample $i$?**

The implementation returns a dictionary containing:

$$
\{\text{image},\text{label}\}.
$$

### Mini-batches

Rather than process all 60,000 samples simultaneously, the DataLoader creates mini-batches of size

$$
B=512.
$$

For one batch:

$$
X\in\mathbb{R}^{B\times784},
$$

$$
y\in\{0,\ldots,9\}^{B}.
$$

### Why mini-batch training?

It balances three goals:

- efficient vectorized computation;
- manageable memory;
- frequent parameter updates.

Full-batch training would use all 60,000 samples per update. Single-sample stochastic training would update after every image. Mini-batches lie between these extremes.

### Why shuffle training but not testing?

Training is shuffled so that consecutive batches do not repeatedly preserve dataset ordering.

Testing does not need shuffling because no learning occurs and order does not affect aggregate accuracy.


## 3. Load and Validate Training and Testing Data

### Shape is part of the mathematical contract

One sample must satisfy

$$
\mathbf{x}\in\mathbb{R}^{784}.
$$

A mini-batch must satisfy

$$
X\in\mathbb{R}^{B\times784}.
$$

The target batch must contain exactly one integer class per image.

### Label domain

For MNIST:

$$
y\in\{0,1,2,3,4,5,6,7,8,9\}.
$$

Cross-Entropy in PyTorch expects integer class indices—not one-hot vectors—in this implementation.

### Why validate the pixel range?

After normalization, every pixel should satisfy

$$
0\le x_j\le1.
$$

A range outside this interval indicates either:

- normalization was not applied;
- data are corrupted;
- an unintended preprocessing step changed the representation.

### Data leakage concept

The code constructs separate training and test loaders. It is important that utilities later receive the correct loader.

The pipeline should conceptually enforce:

$$
\theta
\leftarrow
\operatorname{Train}(D_{train})
$$

then

$$
\operatorname{Evaluate}(\theta,D_{test}).
$$


## 4. Visualize Representative MNIST Samples

### Why visualization belongs in a quantitative pipeline

Before training, humans should inspect what the model actually receives.

A normalized flattened vector can be reshaped only for visualization:

$$
\mathbb{R}^{784}
\rightarrow
\mathbb{R}^{28\times28}.
$$

The model itself still consumes the flat vector.

### What this diagnostic can reveal

Sample visualization catches problems such as:

- images loaded upside down or corrupted;
- label/image mismatch;
- wrong reshape order;
- incorrect intensity scaling;
- unexpected blank samples.

### Important distinction

Visualization is not training.

Calling

$$
\operatorname{imshow}(\mathbf{x})
$$

does not modify the model or dataset. It is an experimental sanity check.

### Technical Implication

Machine learning should not begin with "the loss decreases." It should begin with "the data represent what I believe they represent."

## 5. Define the One-Hidden-Layer MLP Classifier

### Linear layer

For one image vector

$$
\mathbf{x}\in\mathbb{R}^{784},
$$

the first Linear layer computes

$$
\mathbf{z}_1
=
W_1\mathbf{x}+b_1,
$$

where

$$
W_1\in\mathbb{R}^{H\times784},
$$

$$
b_1\in\mathbb{R}^{H}.
$$

For $H=256$, the first layer contains

$$
256\times784+256
$$

trainable parameters.

### Batch Normalization

Before ReLU, the hidden pre-activations are normalized across the current mini-batch.

For hidden feature $j$:

$$
\mu_B
=
\frac1B
\sum_{i=1}^{B}
z_{ij},
$$

$$
\sigma_B^2
=
\frac1B
\sum_{i=1}^{B}
(z_{ij}-\mu_B)^2.
$$

Normalize:

$$
\hat z_{ij}
=
\frac{
z_{ij}-\mu_B
}{
\sqrt{\sigma_B^2+\epsilon}
}.
$$

Then apply trainable scale and shift:

$$
\operatorname{BN}(z_{ij})
=
\gamma_j\hat z_{ij}+\beta_j.
$$

Do not confuse these BatchNorm parameters $\gamma_j,\beta_j$ with unrelated symbols in other modules.

### ReLU

The Rectified Linear Unit is

$$
\operatorname{ReLU}(a)
=
\max(0,a).
$$

Why is it necessary?

Without a nonlinear activation, two Linear layers collapse into one Linear transformation:

$$
W_2(W_1\mathbf{x}+b_1)+b_2
=
W'\mathbf{x}+b'.
$$

So depth would add no nonlinear modeling power.

### Output layer

The second Linear layer produces

$$
\mathbf{z}
=
W_2\mathbf{h}+b_2,
$$

with

$$
\mathbf{z}\in\mathbb{R}^{10}.
$$

These ten numbers are **logits**.

They are not probabilities.

They may be negative, positive, or larger than one.

### Softmax

At inference time, logits become probabilities:

$$
p_k
=
\frac{
e^{z_k}
}{
\sum_{j=1}^{10}e^{z_j}
}.
$$

Then

$$
\sum_{k=1}^{10}p_k=1.
$$

The predicted class is

$$
\hat y
=
\arg\max_k p_k.
$$

Because Softmax is monotonic with respect to logits,

$$
\arg\max_k p_k
=
\arg\max_k z_k.
$$

### Parameter-count intuition

For hidden width $H$, ignoring BatchNorm moment buffers, the main Linear parameters are approximately

$$
784H+H+10H+10.
$$

As $H$ increases, capacity and computational cost increase roughly linearly.


## 6. Verify the Model Architecture and Forward Pass

### Why verify before training for 10 epochs?

Training is expensive compared with one forward pass. A quick architecture check catches mistakes early.

For batch size $B$:

$$
X\in\mathbb{R}^{B\times784}
$$

must produce

$$
Z\in\mathbb{R}^{B\times10}.
$$

### Cross-Entropy loss

For one sample with target class $y$, the categorical Cross-Entropy is

$$
\mathcal{L}
=
-\log p_y.
$$

Using logits directly:

$$
\mathcal{L}
=
-z_y
+
\log
\sum_{k=1}^{10}
e^{z_k}.
$$

For a mini-batch, the implementation uses the mean loss.

### Interpretation

If the model assigns high probability to the correct class,

$$
p_y\rightarrow1,
$$

then

$$
-\log p_y\rightarrow0.
$$

If it assigns tiny probability to the true class, the loss becomes large.

### Finite-loss validation

NaN or infinity in the first pass is a serious warning. Possible causes include:

- invalid inputs;
- numerical overflow;
- wrong tensor dtype;
- malformed architecture.

### Train mode matters

BatchNorm behaves differently in train and evaluation modes.

During training, it uses current batch statistics and updates running statistics.

During evaluation, it uses the stored running estimates.

Therefore explicit

$$
\operatorname{model.train()}
$$

and

$$
\operatorname{model.eval()}
$$

are not optional details.


## 7. Define Training and Evaluation Utilities

### The training loop

For each mini-batch, the implementation performs:

1. move tensors to the target device;
2. clear old gradients;
3. compute logits and loss;
4. backpropagate;
5. update parameters.

In symbols:

$$
\mathbf{z}
=
f_{\theta}(X),
$$

$$
\mathcal{L}
=
\operatorname{CE}(\mathbf{z},y),
$$

$$
\nabla_{\theta}\mathcal{L}
=
\operatorname{Backprop}(\mathcal{L}),
$$

$$
\theta
\leftarrow
\operatorname{AdamUpdate}(\theta,\nabla_{\theta}\mathcal{L}).
$$

### Why zero the gradients?

PyTorch accumulates gradients by default.

Without

$$
\operatorname{optimizer.zero\_grad()},
$$

the current mini-batch gradient would be added to previous gradients.

That is not the intended optimization rule here.

### Backpropagation intuition

The network is a composition:

$$
\mathbf{x}
\rightarrow
\mathbf{z}_1
\rightarrow
\mathbf{h}
\rightarrow
\mathbf{z}
\rightarrow
\mathcal{L}.
$$

The chain rule propagates sensitivity backward:

$$
\frac{\partial\mathcal{L}}{\partial W_2},
\quad
\frac{\partial\mathcal{L}}{\partial W_1},
\quad\ldots
$$

Autograd performs these derivatives automatically, but the mathematics is still the chain rule.

### Adam optimizer

Adam maintains moving estimates of the first and second moments of gradients.

For gradient $g_t$:

$$
m_t
=
\beta_1m_{t-1}
+
(1-\beta_1)g_t,
$$

$$
v_t
=
\beta_2v_{t-1}
+
(1-\beta_2)g_t^2.
$$

After bias correction, the update is approximately

$$
\theta_t
=
\theta_{t-1}
-
\eta
\frac{
\hat m_t
}{
\sqrt{\hat v_t}+\epsilon
}.
$$

The experiment uses

$$
\eta=10^{-3}.
$$

### Weight decay

The optimizer also uses weight decay

$$
10^{-3}.
$$

Its purpose is to discourage excessively large weights and provide regularization.

### Epoch loss

An epoch means one pass through the full training set.

The notebook reports the sample-weighted mean Cross-Entropy over each epoch:

$$
\bar{\mathcal{L}}_{epoch}
=
\frac{
\sum_b B_b\mathcal{L}_b
}{
N_{train}
}.
$$

### Evaluation mode

Evaluation uses:

- $\operatorname{model.eval()}$;
- no gradient tracking;
- the test loader;
- no optimizer steps.

Accuracy is

$$
\operatorname{Accuracy}
=
\frac{
\#\{i:\hat y_i=y_i\}
}{
N_{test}
}.
$$


## 8. Train the 128-, 256-, and 512-Neuron Models

### Controlled experiment

A fair architecture comparison changes one primary factor:

$$
H\in\{128,256,512\}.
$$

Other settings remain fixed:

- same dataset;
- same batch size;
- same epochs;
- same optimizer;
- same learning rate;
- same weight decay;
- same activation;
- same seed reset policy.

This is an experimental-control principle.

### Why reset the seed?

Neural-network initialization is random.

Resetting

$$
\operatorname{torch.manual\_seed}(42)
$$

before each model reduces one source of variability and makes width comparison easier to interpret.

It does not guarantee perfect cross-platform determinism, especially with different hardware/backends.

### Width and model capacity

Increasing $H$ gives the network more hidden features and more parameters.

A wider network can represent more complex mappings.

But

$$
\text{more parameters}
\not\Rightarrow
\text{automatically better test performance}.
$$

Possible reasons include:

- optimization differences;
- overfitting;
- finite training duration;
- random initialization;
- regularization effects.

### Training vs testing signal

Training loss measures fit to the training objective.

Test accuracy measures classification behavior on held-out examples.

They answer different questions.

A model may have lower training loss but nearly identical—or worse—test accuracy.


## 9. Compare Training Loss and Test Accuracy

### Learning curves

For each width, the training history is

$$
\{
\mathcal{L}^{(1)},
\mathcal{L}^{(2)},
\ldots,
\mathcal{L}^{(10)}
\}.
$$

Plotting loss against epoch reveals optimization dynamics.

Typical questions:

- Is loss decreasing?
- Does one model converge faster?
- Has training plateaued?
- Is one curve unstable?

### Final training loss

The final loss is only the last point:

$$
\mathcal{L}^{(10)}.
$$

It should not replace the whole curve.

### Test accuracy

For model $H$:

$$
A_H
=
\frac{
\text{correct test predictions}
}{
10{,}000
}.
$$

Comparing

$$
(\mathcal{L}^{(10)}_H,A_H)
$$

is more informative than either value alone.

### Submitted reference losses

The historical laboratory values are stored only as references:

- 128: $0.0753$
- 256: $0.0344$
- 512: $0.0394$

A new execution may differ because optimization is stochastic and software/hardware behavior can vary.

A difference is not automatically a bug.

### Generalization gap concept

If training loss becomes extremely low while test accuracy stops improving, that can indicate the model is fitting training-specific details.

This lab does not compute train accuracy/test loss curves or a validation split, so conclusions about overfitting must remain limited.


## 10. Visualize Predictions and Confidence Scores

### Prediction is more than one class label

The network outputs ten probabilities:

$$
\mathbf{p}
=
[p_0,p_1,\ldots,p_9].
$$

The predicted digit is

$$
\hat y
=
\arg\max_k p_k,
$$

and confidence is

$$
c
=
\max_k p_k.
$$

### Confidence is not correctness

A model can be:

- correct and highly confident;
- correct and uncertain;
- wrong and uncertain;
- wrong and highly confident.

Therefore

$$
c=0.99
$$

does not prove the prediction is correct.

It only describes the model's own Softmax concentration.

### Full distribution matters

Suppose the true digit is 5.

Model A:

$$
p_5=0.51,
\quad
p_3=0.48.
$$

Model B:

$$
p_5=0.99,
\quad
p_3=0.005.
$$

Both predict 5, but their uncertainty structure is very different.

The bar plot exposes this.

### Softmax caveat

Softmax probability is often called confidence, but neural networks can be miscalibrated.

A perfectly calibrated model would satisfy, approximately:

> among predictions with confidence near 80%, about 80% are correct.

This laboratory does not perform calibration methods such as temperature scaling.


## 11. Compute Confidence-Threshold Precision, Recall, and Accepted Accuracy

### Selective prediction idea

Instead of accepting every prediction, require confidence above threshold $\tau$:

$$
\operatorname{accepted}_i
=
[c_i>\tau].
$$

As $\tau$ increases, the model becomes more selective.

### What is the positive event?

For this analysis, the positive event is **the prediction is correct**:

$$
\operatorname{correct}_i
=
[\hat y_i=y_i].
$$

Then:

$$
TP
=
\#\{
\text{accepted and correct}
\},
$$

$$
FP
=
\#\{
\text{accepted and incorrect}
\},
$$

$$
FN
=
\#\{
\text{rejected but correct}
\}.
$$

### Precision

$$
P
=
\frac{TP}{TP+FP}.
$$

Interpretation:

> Among predictions we choose to accept, what fraction are correct?

This is also often called **selective accuracy**.

### Recall

$$
R
=
\frac{TP}{TP+FN}.
$$

Interpretation:

> Of all predictions that were correct before threshold rejection, what fraction did we retain?

This definition requires $FN$ to count rejected **correct** predictions, not every rejected prediction.

### Accepted accuracy

The notebook additionally reports

$$
A_{accepted}
=
\frac{TP}{N_{test}}.
$$

Interpretation:

> What fraction of the entire test set is both accepted and correct?

This is intentionally different from precision.

### Threshold trade-off

As $\tau$ rises:

- accepted predictions usually decrease;
- precision often rises;
- recall usually falls;
- accepted accuracy cannot exceed the original test accuracy.

This produces a risk/coverage-style trade-off.

### Empty accepted set

At a very high threshold, no predictions may be accepted.

Then

$$
TP+FP=0
$$

and precision is mathematically undefined.

The implementation uses NaN for that case instead of inventing a value.


## 12. Analyze the Best Model and Save Evaluation Curves

### What "best" means here

The submitted laboratory used **lowest final training loss** as its model-selection criterion.

Therefore this notebook preserves:

$$
H^*
=
\arg\min_H
\mathcal{L}^{(10)}_H.
$$

This is a historical/lab criterion, not a universal best-practice model-selection rule.

### Important methodological limitation

A stronger modern workflow would normally use:

- training set for optimization;
- validation set for architecture/hyperparameter selection;
- test set only once for final unbiased evaluation.

This laboratory has train/test data only and no separate validation split.

Therefore the phrase **analysis model** is safer than claiming an optimally selected generalization model.

### Precision–Recall curve

For a sequence of thresholds $\tau$:

$$
(P(\tau),R(\tau)).
$$

The curve shows how correctness purity among accepted predictions trades against retention of correct predictions.

### Accepted-Accuracy–Recall curve

The second curve plots

$$
(A_{accepted}(\tau),R(\tau)).
$$

It shows how the fraction of the full test set that is safely accepted changes as recall changes.

### Why sort by recall for plotting?

Threshold sweeps may produce repeated/non-monotonic coordinates numerically.

Sorting by recall produces a visually coherent horizontal axis without changing the individual metric values.


## 13. Run Numerical and Output-file Validation Checks

### Why "the notebook ran" is not enough

A notebook can execute without raising an exception and still contain invalid results.

Final validation converts assumptions into explicit tests.

### Dataset checks

The expected cardinalities are

$$
60{,}000
$$

and

$$
10{,}000.
$$

### Model-completeness check

The experiment requires exactly

$$
\{128,256,512\}
$$

hidden widths.

If one model failed silently, the comparison would be incomplete.

### Loss-history checks

For each model:

$$
\text{history length}=10
$$

and every loss value must be finite.

NaN loss can arise from numerical instability or corrupted tensors.

### Accuracy bounds

A probability-like accuracy must satisfy

$$
0\le A\le1.
$$

### Prediction completeness

Every model should produce exactly one class prediction for each of the 10,000 test images.

### Confidence validity

Maximum Softmax probabilities must be finite and satisfy

$$
0\le c_i\le1.
$$

### Filesystem validation

The six expected figures are part of the reproducible output:

1. sample batch;
2. training-loss comparison;
3. predictions for 128;
4. predictions for 256;
5. predictions for 512;
6. confidence-threshold curves.

### Deeper expert checks

Beyond automated validation, inspect:

- whether training loss decreases smoothly;
- whether test accuracy is plausible;
- whether prediction examples reveal systematic confusion;
- whether a model is highly confident when wrong;
- whether increasing width changes generalization meaningfully;
- whether conclusions distinguish training and test metrics.

### Validation Interpretation

A trustworthy deep-learning experiment requires agreement between:

$$
\boxed{
\text{data integrity}
+
\text{correct model math}
+
\text{sound optimization}
+
\text{honest metrics}
+
\text{reproducible outputs}
}
$$

## Technical Synthesis

The experiment is structured as a controlled comparison of MLP capacity under a fixed data and optimization protocol:

$$
\boxed{
\text{MNIST data}
\rightarrow
\text{mini-batches}
\rightarrow
\text{MLP}
\rightarrow
\mathcal{L}_{CE}
\rightarrow
\text{optimization}
\rightarrow
\text{test predictions}
\rightarrow
\text{confidence analysis}
}
$$

The interpretation is based on training loss, held-out accuracy, prediction confidence, precision/recall under selective acceptance, and consistency across hidden-layer widths. Architectural comparisons are meaningful only because the data split, preprocessing, training schedule, and evaluation procedure are held fixed.

## Scope and Limitations

### Included

- local MNIST loading;
- normalization to $[0,1]$;
- one-hidden-layer MLP;
- BatchNorm + ReLU;
- Cross-Entropy;
- Adam;
- three hidden widths;
- test accuracy;
- Softmax confidence;
- confidence-threshold analysis;
- visual/numerical validation.

### Not included

- convolutional neural networks;
- spatial feature extraction;
- data augmentation;
- dropout;
- separate validation split;
- hyperparameter search;
- learning-rate scheduling;
- early stopping;
- probability calibration;
- uncertainty estimation.

The notebook should therefore be interpreted as a rigorous study of the **fundamental supervised neural-network pipeline**, not as a state-of-the-art MNIST benchmark.